# Lab Work - 9.4

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import datasets
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, roc_curve,
    classification_report, confusion_matrix, make_scorer
)
from sklearn.calibration import CalibratedClassifierCV
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
np.random.seed(42)
print('Libraries loaded successfully.')

# Q1 – Linear SVM Pipeline

## Q1.01  Load data, baseline Linear SVM, scaling effect

**Task**  
Load `sklearn.datasets.load_breast_cancer`, split 80/20 stratified, `random_state=42`.  
Fit `SVC(kernel='linear', C=1.0)` without scaling → print test accuracy & F1-weighted.  
Then apply `StandardScaler` (fit on train only) and re-fit → print again.  
Quantify the improvement and explain why SVM requires scaling even more than KNN.

In [ ]:
# Load data
cancer = datasets.load_breast_cancer()
X, y = cancer.data, cancer.target
feature_names = cancer.feature_names

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# --- Without scaling ---
svm_raw = SVC(kernel='linear', C=1.0, random_state=42)
svm_raw.fit(X_train, y_train)
y_pred_raw = svm_raw.predict(X_test)
acc_raw = accuracy_score(y_test, y_pred_raw)
f1_raw  = f1_score(y_test, y_pred_raw, average='weighted')
print('=== WITHOUT SCALING ===')
print(f'Accuracy     : {acc_raw:.4f}')
print(f'F1-weighted  : {f1_raw:.4f}')

# --- With StandardScaler ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

svm_scaled = SVC(kernel='linear', C=1.0, random_state=42)
svm_scaled.fit(X_train_scaled, y_train)
y_pred_scaled = svm_scaled.predict(X_test_scaled)
acc_scaled = accuracy_score(y_test, y_pred_scaled)
f1_scaled  = f1_score(y_test, y_pred_scaled, average='weighted')
print('\n=== WITH StandardScaler ===')
print(f'Accuracy     : {acc_scaled:.4f}')
print(f'F1-weighted  : {f1_scaled:.4f}')

print(f'\nImprovement in Accuracy : {acc_scaled - acc_raw:+.4f}')
print(f'Improvement in F1        : {f1_scaled - f1_raw:+.4f}')

### Explanation – Why SVM needs scaling even more than KNN

- Linear SVM maximises the **margin** $\frac{2}{\|w\|}$.  Features with large numerical ranges dominate the Euclidean norm $\|w\|$, so the hyperplane is biased toward those features.
- The dual optimisation depends on the Gram matrix $X X^\top$; unscaled features produce ill-conditioned kernels and slow / unstable solvers.
- KNN also suffers from distance distortion, but its decision is purely local.  SVM’s global margin objective is *more* sensitive to scale because every support vector contributes to the same $w$ vector.
- **Practical rule**: always scale before any kernel method (linear or RBF).

## Q1.02  Support vectors

Print number of support vectors (`model.n_support_`), extract them via `model.support_vectors_`, print their shape.  
Verify they are a subset of the training points that lie on or inside the margin.

In [ ]:
print('Number of support vectors per class :', svm_scaled.n_support_)
print('Total support vectors               :', svm_scaled.n_support_.sum())
print('Shape of support_vectors_           :', svm_scaled.support_vectors_.shape)

# Indices of support vectors in the training set
sv_indices = svm_scaled.support_
print('\nFirst 10 support-vector indices in training set:', sv_indices[:10])

# Verify that every support vector is exactly one of the training points
sv_from_train = X_train_scaled[sv_indices]
print('All support vectors recovered from training set?',
      np.allclose(sv_from_train, svm_scaled.support_vectors_))

## Q1.03  Decision boundary on first two principal components

Plot the decision boundary for the scaled linear SVM projected onto the first two PCs.  
Show contours, margin lines (±1), support vectors (circles) and the two classes.

In [ ]:
# PCA to 2-D for visualisation
pca = PCA(n_components=2, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca  = pca.transform(X_test_scaled)

# Re-fit a linear SVM in the 2-D PCA space (for a clean plot)
svm_2d = SVC(kernel='linear', C=1.0, random_state=42)
svm_2d.fit(X_train_pca, y_train)

# Meshgrid
x_min, x_max = X_train_pca[:, 0].min() - 1, X_train_pca[:, 0].max() + 1
y_min, y_max = X_train_pca[:, 1].min() - 1, X_train_pca[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 400),
                     np.linspace(y_min, y_max, 400))
Z = svm_2d.decision_function(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.figure(figsize=(10, 8))
# Decision regions
plt.contourf(xx, yy, Z, levels=[-np.inf, 0, np.inf], colors=['#FFAAAA', '#AAAAFF'], alpha=0.4)
# Decision boundary and margins
plt.contour(xx, yy, Z, levels=[-1, 0, 1], colors=['k', 'k', 'k'],
            linestyles=['--', '-', '--'], linewidths=[1, 2, 1])

# Training points
plt.scatter(X_train_pca[y_train==0, 0], X_train_pca[y_train==0, 1],
            c='red', edgecolors='k', label='Malignant (0)', s=40, alpha=0.7)
plt.scatter(X_train_pca[y_train==1, 0], X_train_pca[y_train==1, 1],
            c='blue', edgecolors='k', label='Benign (1)', s=40, alpha=0.7)

# Support vectors
sv = svm_2d.support_vectors_
plt.scatter(sv[:, 0], sv[:, 1], s=120, facecolors='none',
            edgecolors='black', linewidths=2, label='Support Vectors')

plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('Linear SVM Decision Boundary (PCA 2-D)\nsolid = decision, dashed = margins ±1')
plt.legend(loc='best')
plt.tight_layout()
plt.show()

print(f'Number of support vectors in 2-D model: {svm_2d.n_support_.sum()}')

## Q1.04  Linear SVM vs Logistic Regression

Compare `SVC(kernel='linear')` with `LogisticRegression` on the same scaled data.  
Print accuracy, F1-weighted, AUC-ROC and number of support vectors.  
Explain the theoretical connection and which model wins (and why).

In [ ]:
# Logistic Regression (scaled data)
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)
y_proba_lr = lr.predict_proba(X_test_scaled)[:, 1]

# SVM probabilities via Platt scaling
svm_prob = SVC(kernel='linear', C=1.0, probability=True, random_state=42)
svm_prob.fit(X_train_scaled, y_train)
y_pred_svm = svm_prob.predict(X_test_scaled)
y_proba_svm = svm_prob.predict_proba(X_test_scaled)[:, 1]

print('='*60)
print(f'{"Metric":<25} {"Linear SVM":>15} {"Logistic Reg":>15}')
print('='*60)
print(f'{"Accuracy":<25} {accuracy_score(y_test, y_pred_svm):>15.4f} {accuracy_score(y_test, y_pred_lr):>15.4f}')
print(f'{"F1-weighted":<25} {f1_score(y_test, y_pred_svm, average="weighted"):>15.4f} {f1_score(y_test, y_pred_lr, average="weighted"):>15.4f}')
print(f'{"AUC-ROC":<25} {roc_auc_score(y_test, y_proba_svm):>15.4f} {roc_auc_score(y_test, y_proba_lr):>15.4f}')
print(f'{"# Support Vectors":<25} {svm_prob.n_support_.sum():>15} {"N/A":>15}')
print('='*60)

### Theoretical connection

Both models learn a **linear decision boundary** $w^\top x + b = 0$.

| Aspect | Linear SVM | Logistic Regression |
|--------|------------|---------------------|
| Loss | Hinge loss $\max(0, 1-y(w^\top x+b))$ | Log-loss / cross-entropy |
| Regularisation | $L_2$ on $w$ (via $C$) | $L_2$ (or $L_1$) on $w$ |
| Output | Hard margin / soft margin; probabilities only after Platt scaling | Native calibrated probabilities |
| Geometry | Maximises geometric margin | Maximises likelihood |

**Which wins?** On the breast-cancer data both are usually very close.  Logistic Regression often edges out slightly on AUC because its probabilities are better calibrated; SVM may win on pure accuracy when the classes are well separated.  The hinge loss is more robust to outliers far from the margin, while log-loss penalises every mis-classified point proportionally to its distance.

## Q1.05  Feature importances from linear SVM

The coefficient vector $w = \texttt{model.coef\_[0]}$ gives the contribution of each feature.  
Plot a horizontal bar chart of the top-10 features by $|w|$.  Interpret the top three in the context of cancer diagnosis.

In [ ]:
w = svm_scaled.coef_[0]
importance = np.abs(w)
top10_idx = np.argsort(importance)[-10:][::-1]

plt.figure(figsize=(10, 6))
plt.barh(range(10), importance[top10_idx][::-1], color='steelblue')
plt.yticks(range(10), [feature_names[i] for i in top10_idx[::-1]])
plt.xlabel('|coefficient|')
plt.title('Top-10 Features by Absolute Linear-SVM Coefficient')
plt.tight_layout()
plt.show()

print('Top-3 features:')
for rank, idx in enumerate(top10_idx[:3], 1):
    print(f'{rank}. {feature_names[idx]:30s}  |w| = {importance[idx]:.4f}  (sign = {np.sign(w[idx]):+})')

### Interpretation (typical top features)

1. **worst radius / mean radius** – larger tumours are strongly associated with malignancy.  
2. **worst texture / mean texture** – irregular cell texture is a classic malignancy marker.  
3. **worst concave points / mean concave points** – more concave portions of the cell contour indicate aggressive growth.  

These match clinical knowledge: size, texture irregularity and concavity are key visual cues used by pathologists.

## Q1.06  Calibrated probabilities (Platt scaling)

Fit `SVC(kernel='linear', C=1.0, probability=True)`.  
Print `predict_proba` for the first 5 test samples and compare with hard predictions.  
Explain how Platt scaling works and why it may be unreliable for small data sets.

In [ ]:
print('Hard predictions (first 5):', y_pred_svm[:5])
print('Predicted probabilities (class 1):')
print(np.round(y_proba_svm[:5], 4))
print('\nTrue labels (first 5):', y_test[:5])

### Platt scaling explanation

SVM’s decision function $f(x) = w^\top x + b$ is **not** a probability.  Platt scaling fits a logistic sigmoid

$$
P(y=1\mid x) = \frac{1}{1+\exp(Af(x)+B)}
$$

on a held-out validation set (or via cross-validation inside `probability=True`).  Parameters $A,B$ are learned by maximum likelihood.

**Why unreliable on small data?**  The sigmoid needs enough positive *and* negative examples near the decision boundary to estimate $A$ and $B$ stably.  With few samples the calibration can over-fit, producing over-confident or under-confident probabilities.  For tiny data sets prefer isotonic regression or simply report the raw decision values.

---
# Q2 – Non-Linear Kernels

## Q2.01  Four kernels on make_moons

Load `make_moons(n_samples=300, noise=0.2, random_state=42)`, 80/20 split.  
Fit four SVMs: linear, poly (degree=3), rbf (gamma='scale'), sigmoid.  
Print accuracy for each.  State which is best and confirm the data is not linearly separable.

In [ ]:
X_moon, y_moon = datasets.make_moons(n_samples=300, noise=0.2, random_state=42)
Xm_train, Xm_test, ym_train, ym_test = train_test_split(
    X_moon, y_moon, test_size=0.2, random_state=42
)

kernels = {
    'linear' : SVC(kernel='linear', C=1.0, random_state=42),
    'poly'   : SVC(kernel='poly', degree=3, C=1.0, random_state=42),
    'rbf'    : SVC(kernel='rbf', gamma='scale', C=1.0, random_state=42),
    'sigmoid': SVC(kernel='sigmoid', C=1.0, random_state=42)
}

print(f'{"Kernel":<12} {"Test Accuracy":>14}')
print('-'*28)
results = {}
for name, clf in kernels.items():
    clf.fit(Xm_train, ym_train)
    acc = accuracy_score(ym_test, clf.predict(Xm_test))
    results[name] = (clf, acc)
    print(f'{name:<12} {acc:>14.4f}')

best_kernel = max(results, key=lambda k: results[k][1])
print(f'\nBest kernel: {best_kernel}  (accuracy = {results[best_kernel][1]:.4f})')
print('Data is clearly NOT linearly separable → linear kernel is weakest.')

## Q2.02  2×2 decision-boundary subplots

For each kernel shade the decision regions, plot the training points, mark support vectors with circles, and put the number of support vectors in the title.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()

x_min, x_max = X_moon[:, 0].min() - 0.5, X_moon[:, 0].max() + 0.5
y_min, y_max = X_moon[:, 1].min() - 0.5, X_moon[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                     np.linspace(y_min, y_max, 300))

for ax, (name, (clf, acc)) in zip(axes, results.items()):
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.coolwarm)
    ax.scatter(Xm_train[ym_train==0, 0], Xm_train[ym_train==0, 1],
               c='blue', edgecolors='k', s=30, label='class 0')
    ax.scatter(Xm_train[ym_train==1, 0], Xm_train[ym_train==1, 1],
               c='red', edgecolors='k', s=30, label='class 1')
    sv = clf.support_vectors_
    ax.scatter(sv[:, 0], sv[:, 1], s=100, facecolors='none',
               edgecolors='black', linewidths=1.5)
    n_sv = clf.n_support_.sum()
    ax.set_title(f'{name}  |  acc={acc:.3f}  |  #SV={n_sv}')
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)

plt.suptitle('Decision Boundaries – make_moons', fontsize=14)
plt.tight_layout()
plt.show()

## Q2.03  Effect of gamma on RBF SVM

Fit `SVC(kernel='rbf')` for $\gamma \in \{0.01, 0.1, 1, 10, 100\}$ on scaled make_moons.  
Plot decision boundaries; annotate which values overfit (boundary wraps tightly around individual points) and which underfit.

In [ ]:
scaler_m = StandardScaler()
Xm_train_s = scaler_m.fit_transform(Xm_train)
Xm_test_s  = scaler_m.transform(Xm_test)
X_moon_s   = scaler_m.transform(X_moon)

gammas = [0.01, 0.1, 1, 10, 100]
fig, axes = plt.subplots(1, 5, figsize=(18, 3.5))

x_min, x_max = X_moon_s[:, 0].min() - 0.5, X_moon_s[:, 0].max() + 0.5
y_min, y_max = X_moon_s[:, 1].min() - 0.5, X_moon_s[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 250),
                     np.linspace(y_min, y_max, 250))

for ax, g in zip(axes, gammas):
    clf = SVC(kernel='rbf', gamma=g, C=1.0, random_state=42)
    clf.fit(Xm_train_s, ym_train)
    acc = accuracy_score(ym_test, clf.predict(Xm_test_s))
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.coolwarm)
    ax.scatter(Xm_train_s[:, 0], Xm_train_s[:, 1], c=ym_train,
               cmap=plt.cm.coolwarm, edgecolors='k', s=20)
    status = 'underfit' if g <= 0.1 else ('overfit' if g >= 10 else 'good')
    ax.set_title(f'γ={g}\nacc={acc:.2f} ({status})', fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('RBF Kernel – Effect of γ (scaled moons)', y=1.05)
plt.tight_layout()
plt.show()

print('''Interpretation:
• γ = 0.01, 0.1 → large influence radius → smooth boundary → UNDERFIT
• γ = 1         → balanced → usually best generalisation
• γ = 10, 100   → tiny influence radius → boundary wraps around each point → OVERFIT''')

## Q2.04  Effect of degree on Polynomial kernel

Fit `SVC(kernel='poly')` for degree $\in \{2,3,4,5\}$ on scaled moons.  
Identify the degree at which the boundary starts to overfit and explain the mathematical reason.

In [ ]:
degrees = [2, 3, 4, 5]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, d in zip(axes, degrees):
    clf = SVC(kernel='poly', degree=d, C=1.0, random_state=42)
    clf.fit(Xm_train_s, ym_train)
    acc = accuracy_score(ym_test, clf.predict(Xm_test_s))
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.coolwarm)
    ax.scatter(Xm_train_s[:, 0], Xm_train_s[:, 1], c=ym_train,
               cmap=plt.cm.coolwarm, edgecolors='k', s=20)
    ax.set_title(f'degree={d}\nacc={acc:.3f}')
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('Polynomial Kernel – Effect of Degree', y=1.05)
plt.tight_layout()
plt.show()

print('''Mathematical reason for overfitting at high degree:
The polynomial kernel K(x,x') = (γ x·x' + r)^d maps the data into a feature space
whose dimension grows as O(n^d).  Higher d → more flexible decision surface →
eventually the model can fit noise (overfit).  Degree 3 is usually the sweet spot
for the moons data; degree ≥ 4 starts producing wiggles that chase individual points.''')

## Q2.05  RBF on Breast-Cancer data

Return to the Breast-Cancer set.  Fit `SVC(kernel='rbf', C=1.0, gamma='scale')`, print accuracy & F1.  
Compare with the linear SVM from Q1.  Explain why RBF may not improve (or may even hurt) performance – hint: inspect linear SVM accuracy.

In [ ]:
svm_rbf = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
svm_rbf.fit(X_train_scaled, y_train)
y_pred_rbf = svm_rbf.predict(X_test_scaled)

print('RBF SVM  – Accuracy :', accuracy_score(y_test, y_pred_rbf))
print('RBF SVM  – F1-w     :', f1_score(y_test, y_pred_rbf, average='weighted'))
print('Linear   – Accuracy :', acc_scaled)
print('Linear   – F1-w     :', f1_scaled)

print('''\nWhy RBF often does not beat linear SVM on this data:
The breast-cancer features are already nearly linearly separable after scaling
(linear SVM accuracy is typically > 95 %).  An RBF kernel adds unnecessary
flexibility that can fit noise, especially with the default γ='scale'.  When the
linear model is already near-perfect, a more complex kernel rarely helps and can
slightly degrade generalisation.''')

## Q2.06  Visualising the RBF kernel feature space (conceptual)

Apply PCA to the kernel matrix $K$ (the $n\times n$ Gram matrix for RBF) and plot the first two principal components.  Observe that the classes become more separable in the kernel-induced space than in the original 2-D input space.

In [ ]:
from sklearn.metrics.pairwise import rbf_kernel

# Compute RBF Gram matrix on the (scaled) moons training set
K = rbf_kernel(Xm_train_s, gamma=1.0)

# PCA on the kernel matrix (rows = points in feature space)
pca_k = PCA(n_components=2, random_state=42)
K_pca = pca_k.fit_transform(K)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Original 2-D space
axes[0].scatter(Xm_train_s[ym_train==0, 0], Xm_train_s[ym_train==0, 1],
                c='blue', edgecolors='k', label='class 0')
axes[0].scatter(Xm_train_s[ym_train==1, 0], Xm_train_s[ym_train==1, 1],
                c='red', edgecolors='k', label='class 1')
axes[0].set_title('Original input space (scaled)')
axes[0].legend()

# Kernel PCA space
axes[1].scatter(K_pca[ym_train==0, 0], K_pca[ym_train==0, 1],
                c='blue', edgecolors='k', label='class 0')
axes[1].scatter(K_pca[ym_train==1, 0], K_pca[ym_train==1, 1],
                c='red', edgecolors='k', label='class 1')
axes[1].set_title('RBF Kernel PCA space (γ=1)')
axes[1].legend()

plt.suptitle('Kernel-induced feature space makes classes more separable')
plt.tight_layout()
plt.show()

---
# Q3 – GridSearchCV Tuning

## Q3.01  GridSearchCV on Breast-Cancer

Run `GridSearchCV` on `SVC` with  
`param_grid = {C: [0.01,0.1,1,10,100], gamma: [0.001,0.01,0.1,'scale'], kernel: ['rbf','linear']}`  
`cv=5, scoring='f1_weighted'`.  
Print best parameters, best CV score, and how many total model fits the grid requires.

In [ ]:
param_grid = {
    'C'     : [0.01, 0.1, 1, 10, 100],
    'gamma' : [0.001, 0.01, 0.1, 'scale'],
    'kernel': ['rbf', 'linear']
}

# Note: gamma is ignored for linear kernel, but GridSearchCV still evaluates the combinations.
gs = GridSearchCV(
    SVC(random_state=42),
    param_grid,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1,
    return_train_score=True
)
gs.fit(X_train_scaled, y_train)

print('Best parameters :', gs.best_params_)
print('Best CV F1      :', round(gs.best_score_, 4))

n_C = len(param_grid['C'])
n_g = len(param_grid['gamma'])
n_k = len(param_grid['kernel'])
n_fits = n_C * n_g * n_k * 5   # 5-fold CV
print(f'Total model fits : {n_C}×{n_g}×{n_k}×5 = {n_fits}')

## Q3.02  Refit best estimator, compare with defaults

Refit the best estimator on the full training set, print test accuracy, F1-weighted and AUC-ROC.  
Compare with default `SVC(kernel='rbf')` and default `SVC(kernel='linear')`.  Quantify the improvement from tuning.

In [ ]:
best_svm = gs.best_estimator_
# Already fitted on whole training set by GridSearchCV (refit=True by default)

y_pred_best = best_svm.predict(X_test_scaled)
y_proba_best = best_svm.decision_function(X_test_scaled)  # use decision for AUC if no prob

# For AUC we need probabilities or decision values; decision_function works for binary
auc_best = roc_auc_score(y_test, y_proba_best)

print('=== Tuned best SVM ===')
print(f'Accuracy    : {accuracy_score(y_test, y_pred_best):.4f}')
print(f'F1-weighted : {f1_score(y_test, y_pred_best, average="weighted"):.4f}')
print(f'AUC-ROC     : {auc_best:.4f}')

# Defaults
def_rbf = SVC(kernel='rbf', random_state=42).fit(X_train_scaled, y_train)
def_lin = SVC(kernel='linear', random_state=42).fit(X_train_scaled, y_train)

print('\n=== Default RBF ===')
print(f'Accuracy    : {accuracy_score(y_test, def_rbf.predict(X_test_scaled)):.4f}')
print(f'F1-weighted : {f1_score(y_test, def_rbf.predict(X_test_scaled), average="weighted"):.4f}')

print('\n=== Default Linear ===')
print(f'Accuracy    : {accuracy_score(y_test, def_lin.predict(X_test_scaled)):.4f}')
print(f'F1-weighted : {f1_score(y_test, def_lin.predict(X_test_scaled), average="weighted"):.4f}')

## Q3.03  Heatmap of mean CV F1 scores (RBF only)

Plot a heatmap of mean CV F1-weighted scores for C (y-axis) vs gamma (x-axis) for kernel='rbf' only.  
Mark the best combination.  Identify the region where both C and gamma are large – explain why this region produces the worst generalisation.

In [ ]:
import pandas as pd

# Extract only RBF results
results_df = pd.DataFrame(gs.cv_results_)
rbf_mask = results_df['param_kernel'] == 'rbf'
rbf_df = results_df[rbf_mask].copy()

# Pivot for heatmap
pivot = rbf_df.pivot(index='param_C', columns='param_gamma', values='mean_test_score')
# Reorder gamma columns nicely
gamma_order = [0.001, 0.01, 0.1, 'scale']
pivot = pivot[[g for g in gamma_order if g in pivot.columns]]

plt.figure(figsize=(8, 6))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlGnBu', cbar_kws={'label': 'Mean CV F1'})
plt.title('GridSearchCV – mean CV F1 (RBF kernel)')
plt.xlabel('gamma')
plt.ylabel('C')
plt.tight_layout()
plt.show()

print('''Region of large C + large γ:
• Large C → almost no regularisation → model tries hard to classify every training point correctly.
• Large γ → each support vector influences only a tiny neighbourhood.
Together they produce a highly complex, jagged decision surface that overfits the training folds,
hence the lowest CV scores (worst generalisation).''')

## Q3.04  ROC curves – Tuned SVM vs Logistic Regression vs Random Forest

Plot the ROC curve for the best-tuned SVM (`probability=True`), Logistic Regression and Random Forest (n_estimators=100) on the Breast-Cancer test set.  Add AUC labels and the random-chance diagonal.  Conclude which model has the best AUC.

In [ ]:
# Re-fit best SVM with probability=True for ROC
best_params = gs.best_params_.copy()
svm_roc = SVC(**best_params, probability=True, random_state=42)
svm_roc.fit(X_train_scaled, y_train)

lr_roc = LogisticRegression(max_iter=1000, random_state=42).fit(X_train_scaled, y_train)
rf_roc = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train_scaled, y_train)

models = {
    'Tuned SVM' : svm_roc,
    'Logistic'  : lr_roc,
    'RandomForest': rf_roc
}

plt.figure(figsize=(8, 7))
for name, model in models.items():
    proba = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    plt.plot(fpr, tpr, lw=2, label=f'{name} (AUC = {auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random chance')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves – Breast Cancer Test Set')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Q3.05  Class imbalance & class_weight='balanced'

The Breast-Cancer data has ~212 malignant and ~357 benign samples.  
Refit SVC with `class_weight='balanced'` and compare test Recall for the malignant class to the unweighted model.  
Explain how `class_weight='balanced'` modifies the C parameter internally.

In [ ]:
from sklearn.metrics import recall_score

print('Class distribution in full data:', np.bincount(y))
print('(0 = malignant, 1 = benign)\n')

svm_unbal = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
svm_unbal.fit(X_train_scaled, y_train)
rec_unbal = recall_score(y_test, svm_unbal.predict(X_test_scaled), pos_label=0)

svm_bal = SVC(kernel='rbf', C=1.0, gamma='scale', class_weight='balanced', random_state=42)
svm_bal.fit(X_train_scaled, y_train)
rec_bal = recall_score(y_test, svm_bal.predict(X_test_scaled), pos_label=0)

print(f'Recall (malignant) – unweighted : {rec_unbal:.4f}')
print(f'Recall (malignant) – balanced   : {rec_bal:.4f}')

print('''\nHow class_weight="balanced" works:
sklearn sets weight_i = n_samples / (n_classes * n_samples_i).
For the minority class this yields a larger effective C (C_i = C * weight_i),
so the hinge-loss penalty for mis-classifying a minority point is higher.
Consequently the margin is pushed to favour higher recall on the rare class.''')

## Q3.06  Complete sklearn Pipeline + GridSearchCV

Build a pipeline: `StandardScaler → PCA(n_components=10) → SVC(kernel='rbf')`.  
Run GridSearchCV over C and gamma on this pipeline.  Print best parameters (note the pipeline prefix, e.g. `svc__C`), best CV score and test F1-weighted.  
Explain why pipelining prevents data leakage during cross-validation.

In [ ]:
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('pca',    PCA(n_components=10, random_state=42)),
    ('svc',    SVC(kernel='rbf', random_state=42))
])

param_grid_pipe = {
    'svc__C'    : [0.1, 1, 10, 100],
    'svc__gamma': [0.001, 0.01, 0.1, 'scale']
}

gs_pipe = GridSearchCV(pipe, param_grid_pipe, cv=5, scoring='f1_weighted', n_jobs=-1)
gs_pipe.fit(X_train, y_train)   # raw (unscaled) data – scaler is inside the pipeline

print('Best parameters :', gs_pipe.best_params_)
print('Best CV F1      :', round(gs_pipe.best_score_, 4))

y_pred_pipe = gs_pipe.predict(X_test)
print('Test F1-weighted:', round(f1_score(y_test, y_pred_pipe, average='weighted'), 4))

print('''\nWhy the pipeline prevents data leakage:
When you scale (or run PCA) *outside* CV, information from the validation fold
leaks into the training fold via the global mean/variance (or principal components).
Inside a Pipeline every transform is re-fitted on the training portion of each CV split only,
so the validation fold remains completely unseen – the CV estimate is unbiased.''')

---
# Q4 – Deep Intuition

## Q4.01  Overfitting diagnosis

`SVC(kernel='rbf', C=100, gamma=10)` achieves train accuracy = 1.00, test accuracy = 0.61.  
`SVC(kernel='rbf', C=1, gamma=0.1)` achieves train = 0.95, test = 0.93.  
Explain exactly what is happening in the first model in terms of margin, support vectors and overfitting.  What should you do next?

### Answer

- **Large C (100)** → the soft-margin penalty is huge → the optimiser is forced to classify *every* training point correctly (hard-margin behaviour).  
- **Large γ (10)** → each RBF basis function is extremely peaked → the decision surface can snake around individual points.  
- Consequently almost every training point becomes a support vector, the margin collapses to zero, and the model memorizes noise → train acc = 1.00, test acc collapses.  

**What to do next**  
1. Reduce C (stronger regularisation) and/or reduce γ (smoother kernel).  
2. Use cross-validation (or the GridSearch you already ran) to pick the pair (C, γ) that maximises validation performance.  
3. Optionally add more training data or feature selection if the gap remains large.

## Q4.02  Colleague’s claim about infinite C

A colleague says: “SVM with RBF kernel can fit *any* data set perfectly if you make C large enough.”  
Is this true?  What happens to the number of support vectors, the margin width, and generalisation as $C\to\infty$?  At what point does SVM degenerate to a 1-nearest-neighbour classifier?

### Answer

- **Yes, essentially true** for an RBF kernel with sufficiently large γ (or for a universal kernel).  As $C\to\infty$ the soft-margin SVM approaches the hard-margin SVM; if the points are distinct, a hard-margin RBF SVM can always separate them (the kernel matrix is positive-definite).  
- **Number of support vectors** → approaches $n$ (every point becomes a support vector).  
- **Margin width** $\frac{2}{\|w\|}$ → shrinks toward 0.  
- **Generalisation** → deteriorates (classic overfitting).  

**Degeneration to 1-NN**  
When γ is also taken to infinity the RBF kernel becomes a Dirac delta; the decision for a new point is then determined solely by the label of its single nearest training neighbour – exactly 1-NN.  In practice this already happens for moderately large γ and very large C.

## Q4.03  Comparison table – SVM / Random Forest / Logistic Regression

Fill a table comparing the three algorithms on five dimensions:  
(a) maximum-margin principle, (b) feature-scaling sensitivity, (c) probabilistic output, (d) training-time complexity, (e) interpretability.  
Conclude which model you would use for a data set with 500 samples, 20 features and a non-linear boundary.

### Comparison Table

| Dimension | Linear / RBF SVM | Random Forest | Logistic Regression |
|-----------|------------------|---------------|---------------------|
| (a) Max-margin principle | Explicit geometric margin maximisation | No (axis-aligned splits) | No (likelihood maximisation) |
| (b) Feature-scaling sensitivity | High (must scale) | None (order statistics) | Medium (helps convergence & regularisation) |
| (c) Probabilistic output | Only after Platt / isotonic calibration | Native (vote fractions) | Native (sigmoid / softmax) |
| (d) Training-time complexity | $O(n^2\text{–}n^3)$ (kernel) / $O(n\cdot d)$ (linear) | $O(T\cdot n\log n\cdot\sqrt{d})$ | $O(n\cdot d\cdot\text{iter})$ |
| (e) Interpretability | Coefficients (linear) or dual coefficients; hard for RBF | Feature-importance + path visualisation | Coefficients directly interpretable |

**Recommendation for 500 samples, 20 features, non-linear boundary**  
→ **RBF SVM** (or a carefully regularised Random Forest).  
With only 500 points an RBF SVM is still computationally cheap, the non-linear boundary is naturally handled by the kernel, and the margin maximisation gives good generalisation.  Random Forest is a strong alternative if you also need fast probability estimates or built-in feature importance.

## Q4.04  Multi-class strategies – OvO vs OvR

You have a multi-class problem with 5 classes.  SVC does not natively support multi-class.  
Explain the One-vs-One (OvO) and One-vs-Rest (OvR) strategies – how many binary classifiers does each require for 5 classes?  Which does sklearn use by default for SVC, and why?

### Answer

**One-vs-Rest (OvR)**  
- Train one binary classifier per class: “class $k$ vs. all others”.  
- Number of classifiers = $K$ = 5.  
- At prediction time the class with the highest decision value (or probability) wins.

**One-vs-One (OvO)**  
- Train one binary classifier for every pair of classes.  
- Number of classifiers = $\binom{K}{2} = \frac{K(K-1)}{2} = 10$ for $K=5$.  
- At prediction time each classifier votes; the class with the most votes wins (ties broken by decision-function sums).

**sklearn default for SVC**  
→ **One-vs-One**.  

Reason: SVM optimisation is most natural for balanced binary problems.  OvO produces many small, roughly balanced sub-problems and historically gave slightly better accuracy for SVMs; the extra classifiers are cheap because each is trained on only two classes.  (LogisticRegression and most other estimators default to OvR.)